# EEG Microstate Tokenisation — Hilbert VQ-VAE (modular descriptors)

Learns a discrete codebook of EEG microstates. Each window is summarised by a
**pluggable Hilbert-based descriptor**, mapped to one of K discrete tokens.

**Descriptor modules** (set `DESCRIPTOR` in the config):
| name | per-window feature | size | sign |
|---|---|---|---|
| `amplitude` | RMS amplitude per (filter, channel) = sqrt(diag R) | `N_filt*C` | >= 0 |
| `cross_spectral` | complex cross-spectral density, upper triangle (real+imag) | `N_filt*C^2` | signed |
| `complex_correlation` | normalised complex correlation, strict upper tri (real+imag) | `N_filt*C*(C-1)` | signed |
| `coherence` | coherence magnitude, strict upper tri | `N_filt*C*(C-1)/2` | >= 0 |

Swapping descriptors changes only one config line — the encoder, quantiser,
decoder, and training loop adapt automatically (the decoder even switches its
output activation between Softplus and linear depending on the descriptor's sign).

**Project layout**
```
project/
  models/
    sinc_convolution.py
    descriptors.py
    sinc_hilbert_encoder.py
    hilbert_vqvae.py
    early_stopping.py
    utils.py
  hilbert_microstate_vqvae.ipynb
```

In [ ]:
import numpy as np
import torch
import torch.optim as optim
import matplotlib.pyplot as plt
import seaborn as sns
from torch.utils.data import DataLoader

from models.hilbert_vqvae import HilbertVQVAE, relative_recon_error
from models.descriptors import DESCRIPTOR_REGISTRY, suggest_freq_low
from models.early_stopping import EarlyStopping
from models.utils import UnsupervisedEEGDataset

print('Available descriptors:', list(DESCRIPTOR_REGISTRY.keys()))

## 1. Configuration

In [ ]:
# ── Parameters you set ───────────────────────────────────────────────
subid = '2022100402_L'
DATA_PATH      = f'/data/1overf/Cleaned_data/{subid}.npy'           # (N_instances, C, T)
FS             = 200                  # sampling frequency (Hz)
CHUNK_SEC      = 2.0                  # window length (seconds)
FILT_DIM       = 145                  # filter kernel length (frequency resolution)

DESCRIPTOR     = 'amplitude'          # 'amplitude' | 'cross_spectral'
                                      # | 'complex_correlation' | 'coherence'

# ── Filterbank: choose ONE of the two modes ──────────────────────────
# Mode A — FIXED bands (recommended for interpretable microstates):
#   filters are frozen to exactly these (low, high) Hz bands, guaranteeing
#   full spectral coverage. N_FILT is set to the number of bands.
FIXED_BANDS = [(4, 8), (8, 12), (12, 20), (20, 30), (30, 45)]   # e.g. theta..gamma
#
# Mode B — LEARNABLE filters: set FIXED_BANDS = None and specify N_FILT and the
#   trainable frequency range [FREQ_LOW, FREQ_HIGH].
N_FILT         = 20                   # used only when FIXED_BANDS is None
FREQ_LOW       = 4.0                  # used only when FIXED_BANDS is None
FREQ_HIGH      = 45.0                 # used only when FIXED_BANDS is None
MIN_BAND       = 2.0
CUTOFF         = 50.0                 # absolute ceiling (e.g. preprocessing bandpass)

CODEBOOK_SIZES = [8, 16, 24, 32, 40, 48, 56, 64]
EXTRACT_OVERLAP_PCT = 0
PRUNE_PCT      = 1.0                  # prune codebook entries used in < this % of
                                      # windows (remap to nearest survivor); 0 disables

CODEBOOK_DIM   = 128  #amplitude 128 coherence 512 
DECODER_HIDDEN = 256  # amplitude 256 coherence 1024
BATCH_SIZE     = 64
LR             = 3e-4
SWEEP_EPOCHS   = 100
FULL_EPOCHS    = 300
PATIENCE       = 20

# ── Derived ───────────────────────────────────────────────────────────
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
data   = np.load(DATA_PATH)
# expected shape (N_instances, C, T). If your array is (N, T, C), 
# I made a mistake so transpose:
data = np.transpose(data, (0, 2, 1))
N_instances, C, T = data.shape
CHUNK_SAMPLES = int(CHUNK_SEC * FS)

# resolve number of filters
if FIXED_BANDS is not None:
    EFFECTIVE_N_FILT = len(FIXED_BANDS)
else:
    EFFECTIVE_N_FILT = N_FILT

_desc_dim = {
    'amplitude':           EFFECTIVE_N_FILT * C,
    'cross_spectral':      EFFECTIVE_N_FILT * C * C,
    'complex_correlation': EFFECTIVE_N_FILT * C * (C - 1),
    'coherence':           EFFECTIVE_N_FILT * (C * (C - 1)) // 2,
}[DESCRIPTOR]

print(f'Device:      {device}')
print(f'Data:        {data.shape}')
print(f'Chunk:       {CHUNK_SEC}s = {CHUNK_SAMPLES} samples')
print(f'Descriptor:  {DESCRIPTOR}  ->  d_desc = {_desc_dim}')
if FIXED_BANDS is not None:
    print(f'Filterbank:  FIXED, {EFFECTIVE_N_FILT} bands: {FIXED_BANDS}')
else:
    print(f'Filterbank:  LEARNABLE, N_FILT={N_FILT}, range [{FREQ_LOW},{FREQ_HIGH}] Hz')

# ── Advisory: minimum reliable frequency for the chosen descriptor ─────
adv = suggest_freq_low(CHUNK_SAMPLES, FS, DESCRIPTOR)
print()
print(f'ADVISORY for "{DESCRIPTOR}": >= {adv["min_cycles"]:.0f} cycles/window '
      f'-> frequencies should be >= {adv["freq_low_min"]:.2f} Hz')
if FIXED_BANDS is not None:
    low_edges = [lo for (lo, hi) in FIXED_BANDS]
    too_low = [b for b in FIXED_BANDS if b[0] < adv['freq_low_min']]
    if too_low:
        print(f'  ** WARNING: these fixed bands start below {adv["freq_low_min"]:.2f} Hz '
              f'and may give unreliable {DESCRIPTOR}: {too_low}')
    else:
        print(f'  All fixed bands satisfy the recommendation.')
else:
    if FREQ_LOW < adv['freq_low_min']:
        print(f'  ** WARNING: FREQ_LOW={FREQ_LOW} < {adv["freq_low_min"]:.2f} Hz — '
              f'low-frequency {DESCRIPTOR} may be unreliable.')
    else:
        print(f'  FREQ_LOW={FREQ_LOW} satisfies the recommendation.')

## 2. Build dataset

Non-overlapping training chunks; instance-level train/val split.

In [ ]:
def chunk_instances(data, chunk_samples):
    N, C, T = data.shape
    starts  = list(range(0, T - chunk_samples + 1, chunk_samples))
    chunks, ids = [], []
    for i in range(N):
        for s in starts:
            chunks.append(data[i, :, s:s + chunk_samples]); ids.append(i)
    return np.array(chunks), np.array(ids)

chunks, instance_ids = chunk_instances(data, CHUNK_SAMPLES)
print(f'Chunks: {chunks.shape}')

np.random.seed(42)
perm       = np.random.permutation(N_instances)
n_val_inst = max(1, int(N_instances * 0.2))
val_inst   = set(perm[:n_val_inst].tolist())
tr_mask    = np.array([i not in val_inst for i in instance_ids])
X_train, X_val = chunks[tr_mask], chunks[~tr_mask]
print(f'Train {len(X_train)} | Val {len(X_val)} chunks')

train_loader = DataLoader(UnsupervisedEEGDataset(X_train), batch_size=BATCH_SIZE, shuffle=True)
val_loader   = DataLoader(UnsupervisedEEGDataset(X_val),   batch_size=BATCH_SIZE, shuffle=False)
full_loader  = DataLoader(UnsupervisedEEGDataset(chunks),  batch_size=BATCH_SIZE, shuffle=False)

## 3. Codebook size sweep

In [ ]:
def build_model(codebook_size):
    return HilbertVQVAE(
        num_EEG_Channels=C, N_filt=N_FILT, Filt_dim=FILT_DIM,
        window_samples=CHUNK_SAMPLES, descriptor=DESCRIPTOR,
        fs=FS, cutoff=CUTOFF,
        freq_low=FREQ_LOW, freq_high=FREQ_HIGH, min_band=MIN_BAND,
        fixed_bands=FIXED_BANDS,
        codebook_size=codebook_size,
        codebook_dim=CODEBOOK_DIM, decoder_hidden=DECODER_HIDDEN,
    ).to(device)


def train_model(model, train_loader, val_loader, n_epochs, patience=PATIENCE,
                checkpoint_path='ckpt.pt',
                collapse_active_frac=0.25, collapse_dist=0.05,
                collapse_patience=5, verbose=True):
    """
    Train with val early stopping AND a sign-independent collapse auto-stop.

    Returns (history, stop_reason) where history is a dict of per-epoch lists:
        'train_loss'  mean training total loss
        'val_loss'    mean validation total loss
        'train_recon' mean training reconstruction loss
        'val_recon'   mean validation reconstruction loss
        'val_err'     mean validation relative reconstruction error
        'active'      active codebook entries
        'dist'        mean normalised pairwise codebook distance
    stop_reason in {'val_loss','collapse','max_epochs'}.
    """
    optimizer = optim.Adam(model.parameters(), lr=LR)
    stopper   = EarlyStopping(patience=patience, path=checkpoint_path)
    history = {k: [] for k in
               ['train_loss','val_loss','train_recon','val_recon','val_err','active','dist']}
    collapse_count, stop_reason = 0, 'max_epochs'

    for epoch in range(n_epochs):
        # ---- train ----
        model.train(); tl = tr = 0.0
        for batch in train_loader:
            batch = batch.to(device)
            _, _, _, losses = model(batch)
            optimizer.zero_grad(); losses['total'].backward(); optimizer.step()
            tl += losses['total'].item(); tr += losses['recon'].item()
        n_tr = len(train_loader)

        # ---- validate ----
        model.eval(); vl = vr = ve = 0.0
        with torch.no_grad():
            for batch in val_loader:
                batch = batch.to(device)
                dh, dt, _, losses = model(batch)
                vl += losses['total'].item(); vr += losses['recon'].item()
                ve += relative_recon_error(dh, dt)
        n_va = len(val_loader)

        diag = model.quantizer.codebook_diagnostics()
        history['train_loss'].append(tl/n_tr)
        history['val_loss'].append(vl/n_va)
        history['train_recon'].append(tr/n_tr)
        history['val_recon'].append(vr/n_va)
        history['val_err'].append(ve/n_va)
        history['active'].append(diag['active_entries'])
        history['dist'].append(diag['mean_norm_dist'])

        if verbose and epoch % 10 == 0:
            print(f'  epoch {epoch:3d} | train {history["train_loss"][-1]:.4f} | '
                  f'val {history["val_loss"][-1]:.4f} | err {history["val_err"][-1]*100:.2f}% | '
                  f'active {diag["active_entries"]}/{model.codebook_size} | '
                  f'dist {diag["mean_norm_dist"]:.3f}')

        # sign-independent collapse
        if diag['active_fraction'] < collapse_active_frac and diag['mean_norm_dist'] < collapse_dist:
            collapse_count += 1
            if collapse_count >= collapse_patience:
                print(f'  collapsed at epoch {epoch}')
                stop_reason='collapse'; break
        else:
            collapse_count = 0

        stopper(history['val_loss'][-1], model)
        if stopper.early_stop: stop_reason='val_loss'; break

    if stop_reason != 'collapse':
        model.load_state_dict(torch.load(checkpoint_path, map_location=device))
    return history, stop_reason

In [ ]:
sweep_errors, sweep_models, sweep_reasons, sweep_histories = {}, {}, {}, {}
for K in CODEBOOK_SIZES:
    print(f'Training K={K}...')
    m = build_model(K)
    hist, reason = train_model(m, train_loader, val_loader, n_epochs=SWEEP_EPOCHS,
                               checkpoint_path=f'ckpt_K{K}.pt', verbose=False)
    sweep_histories[K] = hist
    sweep_errors[K]    = hist['val_err']          # per-epoch val relative error
    sweep_models[K]    = m
    sweep_reasons[K]   = reason
    d = m.quantizer.codebook_diagnostics()
    print(f'  stop={reason} err={min(hist["val_err"])*100:.2f}% '
          f'active={d["active_entries"]}/{K} dist={d["mean_norm_dist"]:.3f} '
          f'cos={d["mean_cosine_sim"]:.3f}')

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
for K, errs in sweep_errors.items():
    style = '--' if sweep_reasons[K]=='collapse' else '-'
    axes[0].plot([e*100 for e in errs], style,
                 label=f'K={K}'+(' [collapsed]' if sweep_reasons[K]=='collapse' else ''))
axes[0].set_xlabel('Epoch'); axes[0].set_ylabel('Val relative error (%)')
axes[0].set_title(f'Training curves — {DESCRIPTOR}'); axes[0].legend(); axes[0].grid(True, alpha=0.3)
best   = [min(sweep_errors[K])*100 for K in CODEBOOK_SIZES]
colors = ['tomato' if sweep_reasons[K]=='collapse' else 'steelblue' for K in CODEBOOK_SIZES]
axes[1].plot(CODEBOOK_SIZES, best, color='steelblue', lw=1.5, zorder=1)
axes[1].scatter(CODEBOOK_SIZES, best, c=colors, s=80, zorder=2)
for K, b in zip(CODEBOOK_SIZES, best):
    axes[1].annotate(f'K={K}\n{b:.2f}%', (K, b), textcoords='offset points',
                     xytext=(0,10), ha='center', fontsize=7)
axes[1].set_xlabel('Codebook size K'); axes[1].set_ylabel('Best val rel error (%)')
axes[1].set_title('Elbow (red = collapsed)'); axes[1].set_xticks(CODEBOOK_SIZES)
axes[1].grid(True, alpha=0.3)
plt.suptitle(f'Codebook size sweep — {DESCRIPTOR}'); plt.tight_layout(); plt.show()

## 4. Select K

In [ ]:
CODEBOOK_SIZE = 16   # <- set after inspecting the elbow
RETRAIN       = False  # True = retrain from scratch for FULL_EPOCHS

if RETRAIN:
    print(f'Retraining K={CODEBOOK_SIZE} for up to {FULL_EPOCHS} epochs...')
    model = build_model(CODEBOOK_SIZE)
    final_history, reason = train_model(model, train_loader, val_loader,
                                        n_epochs=FULL_EPOCHS,
                                        checkpoint_path=f'ckpt_final_K{CODEBOOK_SIZE}.pt',
                                        verbose=True)
    print(f'stop: {reason}')
else:
    model = sweep_models[CODEBOOK_SIZE]
    final_history = sweep_histories[CODEBOOK_SIZE]

print(f'Parameters: {sum(p.numel() for p in model.parameters()):,}')
model.eval()
with torch.no_grad():
    batch = next(iter(val_loader)).to(device)
    dh, dt, tokens, _ = model(batch)
print(f'Relative error: {relative_recon_error(dh, dt)*100:.2f}%')
d = model.quantizer.codebook_diagnostics()
print(f'Active: {d["active_entries"]}/{CODEBOOK_SIZE}  dist {d["mean_norm_dist"]:.3f}  '
      f'cos {d["mean_cosine_sim"]:.3f}')

### 4b. Prune rarely-used codebook entries

Some codebook entries are assigned to very few windows (sometimes only one) --- often
entries that were reset late in training and never re-populated. This step measures
true token usage over the full dataset, keeps only entries used at least
`PRUNE_PCT` percent of the time, and makes the pruned entries unreachable so that
their windows are re-assigned to their nearest surviving entry. Every window stays
tokenised (nothing is dropped) and the total is conserved.

Because this edits the live codebook, **all downstream sections (5, 5b, 6, 7, 8)
automatically operate on the surviving tokens only** --- the visualisations and
saved outputs show the pruned vocabulary. Set `PRUNE_PCT = 0` to disable.

In [ ]:
# ── prune rarely-used codebook entries, remap their windows to nearest survivor ──
# PRUNE_PCT is set in the configuration cell (Section 1).

@torch.no_grad()
def _token_usage(model, loader, device):
    K = model.codebook_size
    counts = torch.zeros(K)
    model.eval()
    for batch in loader:
        z_e, _, _, _ = model.encode(batch.to(device))
        _, tokens, _ = model.quantizer(z_e)
        counts += torch.bincount(tokens, minlength=K).float().cpu()
    return counts

usage_counts = _token_usage(model, full_loader, device)
total_windows = usage_counts.sum().clamp(min=1)
usage_pct = 100.0 * usage_counts / total_windows

# ── usage distribution: see where PRUNE_PCT cuts before pruning anything ──
order = torch.argsort(usage_pct, descending=True)
thr_windows = PRUNE_PCT / 100.0 * float(total_windows)
n_keep = int((usage_pct >= PRUNE_PCT).sum())
print(f'Total windows: {int(total_windows)}   '
      f'PRUNE_PCT={PRUNE_PCT}%  ->  keep entries with >= {thr_windows:.1f} windows')
print(f'{"rank":>4} {"token":>5} {"windows":>8} {"pct":>7}   keep?')
for rank, k in enumerate(order.tolist()):
    cnt = int(usage_counts[k]); pct = float(usage_pct[k])
    if cnt == 0:
        continue  # never-used entries add no information here
    keep = 'keep' if pct >= PRUNE_PCT else 'PRUNE'
    print(f'{rank:>4} {k:>5} {cnt:>8} {pct:>6.2f}%   {keep}')
print(f'-> {n_keep} entries survive, {int((usage_counts > 0).sum()) - n_keep} used-but-pruned, '
      f'{int((usage_counts == 0).sum())} never used')

# quick bar plot of usage with the threshold line
_nz = order[usage_counts[order] > 0]
plt.figure(figsize=(10, 3.5))
_pcts = usage_pct[_nz].cpu().numpy()
_colors = ['#4c72b0' if p >= PRUNE_PCT else '#c44e52' for p in _pcts]
plt.bar(range(len(_nz)), _pcts, color=_colors)
plt.axhline(PRUNE_PCT, color='k', ls='--', lw=1, label=f'PRUNE_PCT = {PRUNE_PCT}%')
plt.xlabel('codebook entry (sorted by usage)'); plt.ylabel('% of windows')
plt.title('Token usage distribution (blue = kept, red = pruned)')
plt.legend(); plt.tight_layout(); plt.show()

if PRUNE_PCT and PRUNE_PCT > 0:
    cb_device = model.quantizer.codebook.device
    survivors = usage_pct >= PRUNE_PCT            # CPU bool mask
    dead_idx  = torch.where(~survivors)[0].to(cb_device)
    surv_idx  = torch.where(survivors)[0].to(cb_device)

    if len(surv_idx) == 0:
        print('No entry meets the threshold; skipping prune (lower PRUNE_PCT).')
        pruned_tokens = []
    elif len(dead_idx) == 0:
        print(f'All {model.codebook_size} entries used >= {PRUNE_PCT}%; nothing to prune.')
        pruned_tokens = []
    else:
        # record where each pruned window WILL go (nearest surviving entry in z_e space
        # = the entry it falls to once the pruned ones are made unreachable)
        cb = model.quantizer.codebook
        d  = torch.cdist(cb[dead_idx], cb[surv_idx])
        remap_target = {int(dead_idx[i]): int(surv_idx[d[i].argmin()]) for i in range(len(dead_idx))}

        # make pruned entries unreachable: they can never win the argmin, so their
        # windows are re-assigned to their nearest surviving entry automatically.
        model.quantizer.codebook[dead_idx]  = 1e9
        model.quantizer.ema_count[dead_idx] = 0.0

        pruned_tokens = dead_idx.cpu().tolist()
        print(f'Pruned {len(pruned_tokens)} / {model.codebook_size} entries '
              f'(used < {PRUNE_PCT}% of {int(total_windows)} windows).')
        print(f'Surviving entries: {surv_idx.tolist()}')

    # verify conservation + that nothing lands on a pruned index
    after = _token_usage(model, full_loader, device)
    assert int(after.sum()) == int(total_windows), 'window count changed!'
    assert after[torch.where(~survivors)[0].cpu()].sum() == 0, 'a pruned entry still used!'
    print(f'Active tokens after prune: {int((after > 0).sum())}  '
          f'(windows conserved: {int(after.sum())})')
    usage_counts = after  # downstream refers to the post-prune usage
else:
    survivors = torch.ones(model.codebook_size, dtype=torch.bool)
    pruned_tokens = []
    print('Pruning disabled (PRUNE_PCT = 0).')

# keep a tidy record for the save-out
prune_info = {
    'prune_pct':      PRUNE_PCT,
    'pruned_tokens':  pruned_tokens,
    'surviving_tokens': torch.where(survivors)[0].tolist(),
    'usage_counts':   usage_counts.int().tolist(),
}

### Training curves for the selected model

Train vs. validation loss over epochs (the standard convergence/overfitting
diagnostic), plus reconstruction error and codebook health across training.

In [ ]:
h = final_history
epochs = range(1, len(h['train_loss']) + 1)

fig, axes = plt.subplots(1, 3, figsize=(18, 5))

# (1) train vs val total loss — the standard curve
axes[0].plot(epochs, h['train_loss'], label='train loss')
axes[0].plot(epochs, h['val_loss'],   label='val loss')
axes[0].set_xlabel('Epoch'); axes[0].set_ylabel('Total loss')
axes[0].set_title(f'Training vs validation loss (K={CODEBOOK_SIZE}, {DESCRIPTOR})')
axes[0].legend(); axes[0].grid(True, alpha=0.3)
# mark the best (early-stopped) epoch
best_ep = int(np.argmin(h['val_loss'])) + 1
axes[0].axvline(best_ep, color='k', ls=':', alpha=0.6)
axes[0].annotate(f'best val\nepoch {best_ep}', (best_ep, min(h['val_loss'])),
                 textcoords='offset points', xytext=(8, 12), fontsize=8)

# (2) validation relative reconstruction error
axes[1].plot(epochs, [e*100 for e in h['val_err']], color='seagreen')
axes[1].set_xlabel('Epoch'); axes[1].set_ylabel('Val relative error (%)')
axes[1].set_title('Reconstruction error'); axes[1].grid(True, alpha=0.3)

# (3) codebook health: active entries and mean distance
ax3 = axes[2]; ax3b = ax3.twinx()
l1 = ax3.plot(epochs, h['active'], color='steelblue', label='active entries')
l2 = ax3b.plot(epochs, h['dist'], color='tomato', label='mean norm dist')
ax3.axhline(CODEBOOK_SIZE, color='steelblue', ls=':', alpha=0.4)
ax3b.axhline(0.05, color='tomato', ls=':', alpha=0.4)  # collapse floor
ax3.set_xlabel('Epoch'); ax3.set_ylabel('Active entries', color='steelblue')
ax3b.set_ylabel('Mean norm dist', color='tomato')
ax3.set_title('Codebook health')
ax3.legend(l1+l2, [x.get_label() for x in l1+l2], loc='center right', fontsize=8)
ax3.grid(True, alpha=0.3)

plt.tight_layout(); plt.show()

## 5. Codebook descriptors

Mean descriptor per token, shape `(K, d_desc)`. The unpacking below adapts to
the chosen descriptor so the visualisation is always meaningful.

In [ ]:
cb_desc, cb_counts = model.compute_codebook_descriptors(full_loader, device=device)
print(f'Codebook descriptors: {cb_desc.shape}   counts: {cb_counts.int().tolist()}')

# frequency labels: exact bands if fixed, else the learned filter centres
if FIXED_BANDS is not None:
    low_f  = np.array([lo for (lo, hi) in FIXED_BANDS])
    high_f = np.array([hi for (lo, hi) in FIXED_BANDS])
    freq_labels = [f'{lo:.0f}-{hi:.0f}Hz' for (lo, hi) in FIXED_BANDS]
else:
    from models.sinc_convolution import constrained_bandpass
    b1, band = (model.encoder.Sinc_Conv.filt_b1.detach().cpu(),
                model.encoder.Sinc_Conv.filt_band.detach().cpu())
    low_f, high_f = constrained_bandpass(b1, band, fs=FS, min_freq=FREQ_LOW,
                                         min_band=MIN_BAND, cutoff=CUTOFF, freq_high=FREQ_HIGH)
    low_f, high_f = low_f.numpy(), high_f.numpy()
    centers = (low_f + high_f) / 2
    freq_labels = [f'{f:.1f}Hz' for f in centers]

In [ ]:
# ── descriptor-aware unpacking helpers ───────────────────────────────
# each returns a dict describing how to lay out one token's descriptor vector

def unpack_amplitude(vec, N_filt, C):
    # (N_filt*C,) -> (N_filt, C)
    return {'kind': 'amplitude', 'data': vec.reshape(N_filt, C)}

def unpack_cross_spectral(vec, N_filt, C):
    # layout: [diag (N_filt*C)] [offreal (N_filt*M)] [offimag (N_filt*M)], M=C(C-1)/2
    M = C*(C-1)//2
    diag = vec[:N_filt*C].reshape(N_filt, C)
    offr = vec[N_filt*C : N_filt*C + N_filt*M].reshape(N_filt, M)
    offi = vec[N_filt*C + N_filt*M :].reshape(N_filt, M)
    return {'kind': 'cross_spectral', 'diag': diag, 'real': offr, 'imag': offi, 'M': M}

def unpack_complex_correlation(vec, N_filt, C):
    M = C*(C-1)//2
    offr = vec[:N_filt*M].reshape(N_filt, M)
    offi = vec[N_filt*M:].reshape(N_filt, M)
    return {'kind': 'complex_correlation', 'real': offr, 'imag': offi, 'M': M}

def unpack_coherence(vec, N_filt, C):
    M = C*(C-1)//2
    return {'kind': 'coherence', 'data': vec.reshape(N_filt, M), 'M': M}

UNPACK = {
    'amplitude': unpack_amplitude,
    'cross_spectral': unpack_cross_spectral,
    'complex_correlation': unpack_complex_correlation,
    'coherence': unpack_coherence,
}

# channel-pair labels for the strict upper triangle
pair_labels = [f'{i}-{j}' for i in range(C) for j in range(i+1, C)]
print(f'{len(pair_labels)} channel pairs:', pair_labels[:6], '...' if len(pair_labels) > 6 else '')

In [ ]:
# visualise each token's descriptor as a (N_filt x feature) heatmap
unpack = UNPACK[DESCRIPTOR]

def token_matrix(vec):
    """Return a (N_filt, n_features) matrix + column labels for plotting."""
    u = unpack(vec, EFFECTIVE_N_FILT, C)
    if u['kind'] == 'amplitude':
        return u['data'].numpy(), [f'Ch{c}' for c in range(C)]
    if u['kind'] == 'coherence':
        return u['data'].numpy(), pair_labels
    if u['kind'] == 'complex_correlation':
        mat = np.concatenate([u['real'].numpy(), u['imag'].numpy()], axis=1)
        return mat, [f'Re {p}' for p in pair_labels] + [f'Im {p}' for p in pair_labels]
    if u['kind'] == 'cross_spectral':
        mat = np.concatenate([u['diag'].numpy(), u['real'].numpy(), u['imag'].numpy()], axis=1)
        cols = [f'P{c}' for c in range(C)] + [f'Re {p}' for p in pair_labels] + [f'Im {p}' for p in pair_labels]
        return mat, cols

ncols = min(CODEBOOK_SIZE, 4)
nrows = int(np.ceil(CODEBOOK_SIZE / ncols))
# constrained_layout=False avoids matplotlib's layout solver stalling when a
# single colorbar is attached to many axes (the cause of the apparent hang).
fig, axes = plt.subplots(nrows, ncols, figsize=(ncols*4, nrows*3.2),
                         squeeze=False, constrained_layout=False)
axes = axes.reshape(-1)
mats = [token_matrix(cb_desc[k])[0] for k in range(CODEBOOK_SIZE)]
vmax = max(np.abs(m).max() for m in mats); vmax = vmax if vmax > 0 else 1.0
vmin = 0 if model.nonnegative else -vmax
cmap = 'inferno' if model.nonnegative else 'RdBu_r'
_, col_labels = token_matrix(cb_desc[0])
im = None
for k in range(CODEBOOK_SIZE):
    ax = axes[k]
    im = ax.imshow(mats[k], aspect='auto', cmap=cmap, vmin=vmin, vmax=vmax)
    ax.set_title(f'k={k} (n={int(cb_counts[k])})', fontsize=8)
    if len(col_labels) <= 12:
        ax.set_xticks(range(len(col_labels))); ax.set_xticklabels(col_labels, fontsize=5, rotation=90)
    else:
        ax.set_xticks([])
    if k % ncols == 0:
        ax.set_yticks(range(EFFECTIVE_N_FILT)); ax.set_yticklabels(freq_labels, fontsize=5)
    else:
        ax.set_yticks([])
for k in range(CODEBOOK_SIZE, len(axes)): axes[k].axis('off')
# reserve space on the right for a colorbar in its OWN axes, instead of stealing
# space from every subplot (fig.colorbar(ax=all_axes) can stall the layout engine).
fig.subplots_adjust(right=0.90)
cax = fig.add_axes([0.92, 0.15, 0.015, 0.7])
fig.colorbar(im, cax=cax, label=DESCRIPTOR)
plt.suptitle(f'Per-token {DESCRIPTOR} descriptors (rows = filters)', fontsize=11)
plt.show()

In [ ]:
# token similarity (works for any descriptor — operates on raw vectors)
norm = cb_desc / cb_desc.norm(dim=-1, keepdim=True).clamp(min=1e-8)
sim  = (norm @ norm.t()).numpy()
plt.figure(figsize=(8,7))
sns.heatmap(sim, annot=True, fmt='.2f', cmap='RdBu_r', center=0, vmin=-1, vmax=1,
            xticklabels=[f'k={k}' for k in range(CODEBOOK_SIZE)],
            yticklabels=[f'k={k}' for k in range(CODEBOOK_SIZE)])
plt.title(f'Token similarity — {DESCRIPTOR}\n(high off-diagonal = redundant)')
plt.tight_layout(); plt.show()

## 5b. Connectivity matrices (for cross_spectral / complex_correlation / coherence)

For the coupling descriptors, each token's descriptor is unpacked back into a set
of `C x C` channel-by-channel matrices, one per frequency band. This is far more
readable than the flat upper-triangle strip when C is large.

- **coherence**: real symmetric matrix in [0,1] per band (diagonal = 1).
- **complex_correlation**: magnitude of the complex matrix per band (diagonal = 1).
- **cross_spectral**: magnitude of the complex cross-spectral matrix per band
  (diagonal = band power).

Run this after section 5 (it reuses `cb_desc`, `freq_labels`, `EFFECTIVE_N_FILT`).
For `amplitude` this section is skipped (amplitude has no cross-channel matrix).

In [ ]:
# ── reconstruct per-band C x C matrices from a token's descriptor vector ──
# The packing must mirror descriptors.py exactly. torch/np triu with offset=1
# enumerates the strict upper triangle in row-major order; we invert that.

def _triu_pairs(C):
    """Row-major (i<j) index pairs, matching torch.triu_indices(C,C,offset=1)."""
    iu = np.triu_indices(C, k=1)
    return iu  # (rows, cols)

def token_band_matrices(vec, descriptor, n_filt, C):
    """
    Return a real array (n_filt, C, C) of per-band connectivity matrices for one
    token descriptor `vec` (1-D numpy array), plus a label for the colour meaning.

    - coherence            -> symmetric, diag set to 1, values in [0,1]
    - complex_correlation  -> |complex corr|, diag set to 1
    - cross_spectral       -> |complex CSD|, diag = band power
    """
    vec = np.asarray(vec)
    M = C * (C - 1) // 2
    iu = _triu_pairs(C)
    mats = np.zeros((n_filt, C, C), dtype=float)

    if descriptor == 'coherence':
        # layout: [n_filt * M] real, band-major
        block = vec.reshape(n_filt, M)
        for f in range(n_filt):
            m = np.zeros((C, C)); m[iu] = block[f]; m = m + m.T
            np.fill_diagonal(m, 1.0)
            mats[f] = m
        return mats, 'coherence (0-1)'

    if descriptor == 'complex_correlation':
        # layout: [n_filt*M real][n_filt*M imag], diagonal dropped (==1)
        re = vec[:n_filt*M].reshape(n_filt, M)
        im = vec[n_filt*M:].reshape(n_filt, M)
        for f in range(n_filt):
            mr = np.zeros((C, C)); mr[iu] = re[f]
            mi = np.zeros((C, C)); mi[iu] = im[f]
            mag = np.zeros((C, C)); mag[iu] = np.hypot(re[f], im[f])
            mag = mag + mag.T
            np.fill_diagonal(mag, 1.0)
            mats[f] = mag
        return mats, '|complex correlation| (0-1)'

    if descriptor == 'cross_spectral':
        # layout: [n_filt*C diag real][n_filt*M off real][n_filt*M off imag]
        diag = vec[:n_filt*C].reshape(n_filt, C)
        re   = vec[n_filt*C : n_filt*C + n_filt*M].reshape(n_filt, M)
        im   = vec[n_filt*C + n_filt*M :].reshape(n_filt, M)
        for f in range(n_filt):
            mag = np.zeros((C, C)); mag[iu] = np.hypot(re[f], im[f])
            mag = mag + mag.T
            np.fill_diagonal(mag, diag[f])
            mats[f] = mag
        return mats, '|cross-spectral density|'

    raise ValueError(f"{descriptor} has no connectivity-matrix view (use section 5).")

In [ ]:
# ── plot: one figure per token, a row of C x C matrices across bands ──────
if DESCRIPTOR == 'amplitude':
    print("amplitude has no connectivity matrix — see section 5 heatmaps instead.")
else:
    # choose which tokens to show: all active (populated) tokens
    active_tokens = [k for k in range(CODEBOOK_SIZE) if cb_counts[k] > 0]

    # shared colour scale across all tokens/bands for comparability
    all_mats = []
    for k in active_tokens:
        mats, clabel = token_band_matrices(cb_desc[k].numpy(), DESCRIPTOR, EFFECTIVE_N_FILT, C)
        all_mats.append(mats)
    all_mats = np.stack(all_mats)                      # (n_active, n_filt, C, C)
    # for [0,1] descriptors fix vmin/vmax; for CSD use data range
    if DESCRIPTOR in ('coherence', 'complex_correlation'):
        vmin, vmax = 0.0, 1.0
    else:
        vmin, vmax = 0.0, np.percentile(all_mats, 99)

    nbands = EFFECTIVE_N_FILT
    for idx, k in enumerate(active_tokens):
        fig, axes = plt.subplots(1, nbands, figsize=(2.1*nbands, 2.4))
        if nbands == 1: axes = [axes]
        for f in range(nbands):
            ax = axes[f]
            im = ax.imshow(all_mats[idx, f], vmin=vmin, vmax=vmax, cmap='viridis')
            ax.set_title(freq_labels[f], fontsize=8)
            ax.set_xticks([]); ax.set_yticks([])
            if f == 0:
                ax.set_ylabel(f'k={k}\n(n={int(cb_counts[k])})', fontsize=8)
        fig.colorbar(im, ax=axes, shrink=0.7, label=clabel)
        plt.suptitle(f'Token k={k} — {DESCRIPTOR} per band  (channel x channel)', fontsize=10)
        plt.show()

In [ ]:
# ── compact overview: band-averaged C x C matrix per token, in one grid ───
if DESCRIPTOR == 'amplitude':
    print("amplitude has no connectivity matrix — see section 5.")
else:
    active_tokens = [k for k in range(CODEBOOK_SIZE) if cb_counts[k] > 0]
    ncols = min(len(active_tokens), 4)
    nrows = int(np.ceil(len(active_tokens) / ncols))
    fig, axes = plt.subplots(nrows, ncols, figsize=(ncols*3, nrows*3))
    axes = np.array(axes).reshape(-1)
    if DESCRIPTOR in ('coherence', 'complex_correlation'):
        vmin, vmax = 0.0, 1.0
    else:
        vmin, vmax = None, None
    for ax_i, k in enumerate(active_tokens):
        mats, clabel = token_band_matrices(cb_desc[k].numpy(), DESCRIPTOR, EFFECTIVE_N_FILT, C)
        avg = mats.mean(axis=0)                        # average over bands
        im = axes[ax_i].imshow(avg, vmin=vmin, vmax=vmax, cmap='viridis')
        axes[ax_i].set_title(f'k={k} (n={int(cb_counts[k])})', fontsize=8)
        axes[ax_i].set_xticks([]); axes[ax_i].set_yticks([])
    for j in range(len(active_tokens), len(axes)): axes[j].axis('off')
    fig.colorbar(im, ax=axes.tolist(), shrink=0.6, label=f'{clabel}, band-avg')
    plt.suptitle(f'Band-averaged {DESCRIPTOR} connectivity per token (channel x channel)', fontsize=11)
    plt.show()

## 6. Token representation of the data

Re-express every instance as a sequence of codebook token ids (one token id per
window), with the sample offset of each window. This is the discrete
representation of the data in the learned codebook.

In [ ]:
# Re-express every instance as a sequence of codebook token ids (one per window).
all_seqs, all_offsets = [], []
for i in range(N_instances):
    toks, offs = model.tokenise_recording(
        data[i], extract_overlap_pct=EXTRACT_OVERLAP_PCT, device=device)
    all_seqs.append(toks.numpy().astype(np.int64))
    all_offsets.append(offs.astype(np.int64))

seq_lengths = np.array([len(t) for t in all_seqs])
print(f'Instances: {N_instances}')
print(f'Tokens per instance: min={seq_lengths.min()}, max={seq_lengths.max()}, '
      f'total={seq_lengths.sum()}')

# raster of one instance's token sequence over time
fig, ax = plt.subplots(figsize=(12, 3))
ax.scatter(all_offsets[0] / FS, all_seqs[0], c=all_seqs[0], cmap='tab20', s=6)
ax.set_xlabel('Time (s)'); ax.set_ylabel('Token id')
ax.set_title('Token sequence — instance 0')
plt.tight_layout(); plt.show()

## 7. Learned Sinc filter bands

In [ ]:
plt.figure(figsize=(8,5))
for i, (lo, hi) in enumerate(zip(low_f, high_f)):
    plt.plot([lo.item(), hi.item()], [i, i], 'r-', lw=2)
plt.xlabel('Frequency (Hz)'); plt.ylabel('Filter index'); plt.title('Filter bands (fixed)' if FIXED_BANDS is not None else 'Learned Sinc filter bands')
plt.xticks(np.arange(0, CUTOFF + 5, 5)); plt.grid(True, alpha=0.3, axis='x'); plt.tight_layout(); plt.show()

## 8. Save model, statistics, codebook analysis, and token representation

Writes:
- **`<prefix>_model.pt`** — trained `HilbertVQVAE` state dict plus a config dict
  (descriptor, K, bands, fs, channels, window, frequency bounds, …) so the model
  can be reloaded exactly.
- **`<prefix>_stats.npz`** — fitting/testing statistics: per-epoch training history
  and final train vs. validation metrics (reported separately).
- **`<prefix>_codebook.npz`** — descriptive analysis of the codebook: the per-token
  mean descriptor and counts, plus the unpacked interpretable form (amplitude maps
  or per-band C×C connectivity matrices), with band labels.
- **`<prefix>_tokens.npz`** — the new representation of the data: one token id per
  window for every instance, with window offsets.

All `.npz` files are self-describing and can be loaded and visualised on their own.


In [ ]:
import os, json as _json

SAVE_PREFIX = f'{subid}_' + DESCRIPTOR + f'_K{CODEBOOK_SIZE}'
SAVE_DIR    = '.'   # change if you want a subfolder
os.makedirs(SAVE_DIR, exist_ok=True)

# ---- 1. model (.pt): state dict + config for exact reload ----------------
model_config = {
    'descriptor':       DESCRIPTOR,
    'codebook_size':    CODEBOOK_SIZE,
    'num_EEG_Channels': C,
    'N_filt':           EFFECTIVE_N_FILT,
    'Filt_dim':         FILT_DIM,
    'window_samples':   CHUNK_SAMPLES,
    'fs':               FS,
    'cutoff':           CUTOFF,
    'freq_low':         FREQ_LOW,
    'freq_high':        FREQ_HIGH,
    'min_band':         MIN_BAND,
    'fixed_bands':      FIXED_BANDS,
    'codebook_dim':     CODEBOOK_DIM,
    'decoder_hidden':   DECODER_HIDDEN,
    'chunk_sec':        CHUNK_SEC,
    'extract_overlap_pct': EXTRACT_OVERLAP_PCT,
}
model_path = os.path.join(SAVE_DIR, SAVE_PREFIX + '_model.pt')
torch.save({'state_dict': model.state_dict(), 'config': model_config}, model_path)
print('saved', model_path)

# to reload later:
#   ckpt = torch.load(model_path, map_location='cpu')
#   m = HilbertVQVAE(num_EEG_Channels=ckpt['config']['num_EEG_Channels'],
#                    N_filt=ckpt['config']['N_filt'], Filt_dim=ckpt['config']['Filt_dim'],
#                    window_samples=ckpt['config']['window_samples'],
#                    descriptor=ckpt['config']['descriptor'], fs=ckpt['config']['fs'],
#                    cutoff=ckpt['config']['cutoff'], freq_low=ckpt['config']['freq_low'],
#                    freq_high=ckpt['config']['freq_high'], min_band=ckpt['config']['min_band'],
#                    fixed_bands=ckpt['config']['fixed_bands'],
#                    codebook_size=ckpt['config']['codebook_size'],
#                    codebook_dim=ckpt['config']['codebook_dim'],
#                    decoder_hidden=ckpt['config']['decoder_hidden'])
#   m.load_state_dict(ckpt['state_dict'])

In [ ]:
# ---- 2. statistics (.npz): training history + final train/val metrics -----

# recompute final relative error separately on train and val splits
@torch.no_grad()
def _split_metrics(loader):
    model.eval(); err = tot = rec = 0.0; n = 0
    for batch in loader:
        batch = batch.to(device)
        dh, dt, _, losses = model(batch)
        err += relative_recon_error(dh, dt)
        tot += losses['total'].item()
        rec += losses['recon'].item()
        n   += 1
    return err/n, tot/n, rec/n

train_err, train_tot, train_rec = _split_metrics(train_loader)
val_err,   val_tot,   val_rec   = _split_metrics(val_loader)
diag = model.quantizer.codebook_diagnostics()

print(f'TRAIN  rel_err={train_err*100:.2f}%  total={train_tot:.4f}  recon={train_rec:.4f}')
print(f'VAL    rel_err={val_err*100:.2f}%  total={val_tot:.4f}  recon={val_rec:.4f}')
print(f'codebook: active={diag["active_entries"]}/{CODEBOOK_SIZE}  '
      f'dist={diag["mean_norm_dist"]:.3f}')

stats_path = os.path.join(SAVE_DIR, SAVE_PREFIX + '_stats.npz')
np.savez(
    stats_path,
    # per-epoch history (arrays)
    hist_train_loss = np.array(final_history['train_loss']),
    hist_val_loss   = np.array(final_history['val_loss']),
    hist_train_recon= np.array(final_history['train_recon']),
    hist_val_recon  = np.array(final_history['val_recon']),
    hist_val_err    = np.array(final_history['val_err']),
    hist_active     = np.array(final_history['active']),
    hist_dist       = np.array(final_history['dist']),
    # final split metrics (scalars)
    final_train_rel_err = train_err,
    final_val_rel_err   = val_err,
    final_train_total   = train_tot,
    final_val_total     = val_tot,
    final_train_recon   = train_rec,
    final_val_recon     = val_rec,
    active_entries      = diag['active_entries'],
    mean_norm_dist      = diag['mean_norm_dist'],
    mean_cosine_sim     = diag['mean_cosine_sim'],
    ema_count           = diag['ema_count'].numpy(),
    usage_fraction      = diag['usage_fraction'].numpy(),
    # sweep summary across all K (best val err per K)
    sweep_K        = np.array(CODEBOOK_SIZES),
    sweep_best_err = np.array([min(sweep_errors[K]) for K in CODEBOOK_SIZES]),
    # config echoed for standalone use
    descriptor = DESCRIPTOR, codebook_size = CODEBOOK_SIZE,
    fs = FS, n_channels = C, n_filt = EFFECTIVE_N_FILT,
)
print('saved', stats_path)

In [ ]:
# ---- 3. codebook analysis (.npz): per-token descriptors + interpretable form

# raw per-token mean descriptor and counts (from section 5)
cb_desc_np   = cb_desc.numpy()          # (K, d_desc)
cb_counts_np = cb_counts.numpy()        # (K,)
band_low  = np.array(low_f,  dtype=float)
band_high = np.array(high_f, dtype=float)

cb_path = os.path.join(SAVE_DIR, SAVE_PREFIX + '_codebook.npz')
save_kwargs = dict(
    descriptor      = DESCRIPTOR,
    codebook_size   = CODEBOOK_SIZE,
    n_filt          = EFFECTIVE_N_FILT,
    n_channels      = C,
    counts          = cb_counts_np,
    descriptor_raw  = cb_desc_np,                 # (K, d_desc) flat
    band_low        = band_low,
    band_high       = band_high,
    band_labels     = np.array(freq_labels),
    # codebook vectors in the D-dim quantiser space (pruned entries set to 0 below)
    codebook_vectors = model.quantizer.codebook.detach().cpu().numpy(),  # (K, D)
    # ── pruning record (Section 4b) ──
    prune_pct        = prune_info['prune_pct'],
    pruned_tokens    = np.array(prune_info['pruned_tokens'],    dtype=int),
    surviving_tokens = np.array(prune_info['surviving_tokens'], dtype=int),
    usage_counts     = np.array(prune_info['usage_counts'],     dtype=int),
)
# pruned entries were made unreachable (set to 1e9); zero them in the saved
# vectors so the file isn't polluted with sentinel values.
if len(prune_info['pruned_tokens']) > 0:
    save_kwargs['codebook_vectors'][np.array(prune_info['pruned_tokens'], dtype=int)] = 0.0

# unpacked, interpretable form per descriptor
if DESCRIPTOR == 'amplitude':
    # (K, N_filt, C) amplitude maps
    amp_maps = cb_desc_np.reshape(CODEBOOK_SIZE, EFFECTIVE_N_FILT, C)
    save_kwargs['amplitude_maps'] = amp_maps
    print('amplitude_maps:', amp_maps.shape)
else:
    # (K, N_filt, C, C) connectivity matrices via the section-5b reconstructor
    mats_all = np.zeros((CODEBOOK_SIZE, EFFECTIVE_N_FILT, C, C), dtype=float)
    for k in range(CODEBOOK_SIZE):
        if cb_counts_np[k] > 0:
            m, _ = token_band_matrices(cb_desc_np[k], DESCRIPTOR, EFFECTIVE_N_FILT, C)
            mats_all[k] = m
    save_kwargs['connectivity_matrices'] = mats_all
    print('connectivity_matrices:', mats_all.shape)

np.savez(cb_path, **save_kwargs)
print('saved', cb_path)

In [ ]:
# ---- 4. token representation (.npz): one token id per window per instance ---

# instances can have different numbers of windows -> store as object arrays,
# plus a flat concatenation with an index for easy vectorised analysis.
tokens_obj  = np.empty(N_instances, dtype=object)
offsets_obj = np.empty(N_instances, dtype=object)
for i in range(N_instances):
    tokens_obj[i]  = all_seqs[i]
    offsets_obj[i] = all_offsets[i]

# flat form: all tokens concatenated, with instance id per token
flat_tokens   = np.concatenate(all_seqs)             if N_instances else np.array([], np.int64)
flat_offsets  = np.concatenate(all_offsets)          if N_instances else np.array([], np.int64)
flat_instance = np.concatenate([np.full(len(all_seqs[i]), i, np.int64)
                                for i in range(N_instances)]) if N_instances else np.array([], np.int64)

tok_path = os.path.join(SAVE_DIR, SAVE_PREFIX + '_tokens.npz')
np.savez(
    tok_path,
    tokens_per_instance  = tokens_obj,      # object array of (n_win,) int arrays
    offsets_per_instance = offsets_obj,     # object array of (n_win,) int arrays
    seq_lengths          = seq_lengths,
    flat_tokens          = flat_tokens,     # (total_windows,)
    flat_offsets         = flat_offsets,    # (total_windows,)
    flat_instance_id     = flat_instance,   # (total_windows,)
    codebook_size        = CODEBOOK_SIZE,
    fs                   = FS,
    extract_overlap_pct  = EXTRACT_OVERLAP_PCT,
    surviving_tokens     = np.array(prune_info['surviving_tokens'], dtype=int),
    prune_pct            = prune_info['prune_pct'],
)
# object arrays require allow_pickle=True on load
print('saved', tok_path)
print('  (load token arrays with np.load(path, allow_pickle=True))')

print('\nAll outputs written with prefix:', SAVE_PREFIX)